<a href="https://colab.research.google.com/github/MINEGHOST007/RecommenderSystem/blob/main/all.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#1
person_req = []
n = int(input("Enter No of Products : "))
i = 0
while i<n:
  person_req.append(int(input(f"Enter the requirement of {i+1} th product: ")))
  i+=1
options = []
k = 2
i = 0
while i<k:
  j = 0
  availability = []
  while j<n:
    availability.append(int(input(f"Enter the Quantity of product {j+1} in 1 unit of {i+1} th option : ")))
    j+=1
  options.append(availability)
  i+=1
costs = []
i = 0
while i<k:
  costs.append(int(input(f"Enter the price of 1 unit of {i+1} th option : ")))
  i+=1
import matplotlib.pyplot as plt
import numpy as np

xi = [person_req[i] / options[0][i] for i in range(n)]
yi = [person_req[i] / options[1][i] for i in range(n)]

x_vals = np.linspace(0, max(xi)*1.2, 400)
plt.figure()

X, Y = np.meshgrid(x_vals, x_vals)
feas = np.ones_like(X, dtype=bool)
for i in range(n):
    feas &= (options[0][i]*X + options[1][i]*Y >= person_req[i])
feas &= (X>=0)&(Y>=0)
plt.contourf(X, Y, feas, levels=[-1,0,1], colors=['white','lightgreen'], alpha=0.5)

for i in range(n):
    plt.plot([xi[i], 0], [0, yi[i]], label=f'constraint product {i+1}')

pts = []

def is_feasible(x, y):
    if x < 0 or y < 0:
        return False
    return all(options[0][i]*x + options[1][i]*y >= person_req[i]
               for i in range(n))

for i in range(n):
    x_int = xi[i]
    if is_feasible(x_int, 0):
        pts.append((x_int, 0))

    y_int = yi[i]
    if is_feasible(0, y_int):
        pts.append((0, y_int))

for a in range(n):
    for b in range(a+1, n):
        A = np.array([[options[0][a], options[1][a]],
                      [options[0][b], options[1][b]]])
        bvec = np.array([person_req[a], person_req[b]])
        det = np.linalg.det(A)
        if abs(det) < 1e-9:
            continue
        x_sol, y_sol = np.linalg.solve(A, bvec)
        if is_feasible(x_sol, y_sol):
            pts.append((x_sol, y_sol))

def Z(pt):
    return costs[0]*pt[0] + costs[1]*pt[1]

best = min(pts, key=Z)
best_cost = Z(best)
print("Optimal solution:", best, "with cost", best_cost)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linprog

def solve_hat_lpp(labor_total,h1_labor,h2_labor,max_h1, max_h2,profit_h1, profit_h2):
    c = [-profit_h1, -profit_h2]

    A = [
        [h1_labor, h2_labor],
        [1, 0],
        [0, 1]
    ]
    b = [labor_total, max_h1, max_h2]

    x_bounds = (0, None)
    y_bounds = (0, None)

    res = linprog(c, A_ub=A, b_ub=b, bounds=[x_bounds, y_bounds], method='highs')

    if res.success:
        x, y = res.x
        max_profit = -res.fun
        print(f"Optimal solution:")
        print(f"  Hat H1 (x) = {x:.2f}")
        print(f"  Hat H2 (y) = {y:.2f}")
        print(f"  Maximum Profit = ₹{max_profit:.2f}")
    else:
        print("No optimal solution found.")

    x_vals = np.linspace(0, max_h1 + 50, 500)
    y1 = (labor_total - h1_labor * x_vals) / h2_labor
    y2 = [max_h2] * len(x_vals)
    y3 = np.maximum(0, np.zeros_like(x_vals))

    plt.figure(figsize=(10, 6))
    plt.plot(x_vals, y1, label=f'{h1_labor}x + {h2_labor}y ≤ {labor_total}')
    plt.axvline(max_h1, color='g', linestyle='--', label=f'x ≤ {max_h1}')
    plt.axhline(max_h2, color='m', linestyle='--', label=f'y ≤ {max_h2}')
    plt.fill_between(x_vals, 0, np.minimum.reduce([y1, y2]), where=(x_vals <= max_h1), color='skyblue', alpha=0.4)

    if res.success:
        plt.plot(x, y, 'ro', label='Optimal Point')
        plt.text(x, y, f"  ({x:.1f}, {y:.1f})", fontsize=12)
    plt.xlim(0, max_h1 + 50)
    plt.ylim(0, max_h2 + 50)
    plt.xlabel("H1 hats (x)")
    plt.ylabel("H2 hats (y)")
    plt.title("Graphical Solution of Hat Production Problem")
    plt.legend()
    plt.grid(True)
    plt.show()

solve_hat_lpp(500,2,1,150,250,8,5)

In [ ]:
#2
import numpy as np

def north_west_corner(supply, demand, cost):
    m, n = len(supply), len(demand)
    if sum(supply) != sum(demand):
        raise ValueError("Unbalanced transportation instance")

    S, D = supply[:], demand[:]
    X = np.zeros((m, n), dtype=int)
    i = j = 0
    used = 0
    total = 0

    take = lambda ii, jj: min(S[ii], D[jj])
    advance = lambda ii, jj: (ii + (S[ii] == 0), jj + (D[jj] == 0))

    while i < m and j < n:
        q = take(i, j)
        if q:
            X[i, j] = q
            total += q * cost[i][j]
            used += 1
        S[i] -= q
        D[j] -= q
        i, j = advance(i, j)

    need = m + n - 1
    return X, total, {"occupied": used, "required": need, "degenerate": used < need}


if __name__ == "__main__":
    try:
        supply = list(map(int, input("Supply (comma-separated): ").split(",")))
        demand = list(map(int, input("Demand (comma-separated): ").split(",")))
        m, n = len(supply), len(demand)

        print(f"Enter {m} rows of cost matrix:")
        costs = [list(map(int, input().split(","))) for _ in range(m)]

        #22CS8179 #Kinnareash
        X, z, info = north_west_corner(supply, demand, costs)
        print("\nAllocation Matrix:\n", X)
        print(f"\nTotal Cost: {z}")
        print("Degenerate" if info["degenerate"] else "Non-degenerate")
    except Exception as e:
        print("Error:", e)

In [ ]:
#3
import numpy as np
# Balanced Suppy & Demand # 22CS8011 # Sankeerth
def least_cost_method(supply, demand, cost):
    m, n = len(supply), len(demand)
    if sum(supply) != sum(demand):
      raise ValueError("Unbalanced transportation instance")

    S, D = supply[:], demand[:]
    C = [row[:] for row in cost]
    X = np.zeros((m, n), dtype=int)
    used = 0
    total_cost = 0

    while sum(S) > 0 and sum(D) > 0:
        min_cost = float('inf')
        min_i, min_j = -1, -1
        for i in range(m):
            for j in range(n):
                if S[i] > 0 and D[j] > 0 and C[i][j] < min_cost:
                    min_cost = C[i][j]
                    min_i, min_j = i, j
        if min_i == -1:
            break
        allocation = min(S[min_i], D[min_j])
        X[min_i, min_j] = allocation
        total_cost += allocation * C[min_i][min_j]
        S[min_i] -= allocation
        D[min_j] -= allocation
        used += 1
        if S[min_i] == 0:
            for j in range(n):
                C[min_i][j] = float('inf')
        if D[min_j] == 0:
            for i in range(m):
                C[i][min_j] = float('inf')


    need = m + n - 1
    return X, total_cost, {"occupied": used, "required": need, "degenerate": used < need}

if __name__ == "__main__":
    try:
        supply = list(map(int, input("Supply (comma-separated): ").split(",")))
        demand = list(map(int, input("Demand (comma-separated): ").split(",")))

        m_initial = len(supply)
        n_initial = len(demand)

        print(f"Enter {m_initial} rows of cost matrix:")
        costs = [list(map(int, input().split(","))) for _ in range(m_initial)]

        #22CS8011 #Sankeerth
        X, z, info = least_cost_method(supply, demand, costs)
        print("\nAllocation Matrix:\n", X)
        print(f"\nTotal Cost: {z}")
        print("Degenerate" if info["degenerate"] else "Non-degenerate")
    except Exception as e:
        print("Error:", e)

In [ ]:



import numpy as np
# Unbalanced Suppy & Demand # 22CS8011 # Sankeerth
def least_cost_method(supply, demand, cost):
    m, n = len(supply), len(demand)
    total_supply = sum(supply)
    total_demand = sum(demand)
    if total_supply < total_demand:
        supply.append(total_demand - total_supply)
        cost.append([0] * n)
        m += 1
    elif total_demand < total_supply:
        demand.append(total_supply - total_demand)
        for row in cost:
            row.append(0)
        n += 1

    S, D = supply[:], demand[:]
    C = [row[:] for row in cost]
    X = np.zeros((m, n), dtype=int)
    used = 0
    total_cost = 0
    while sum(S) > 0 and sum(D) > 0:
        min_cost = float('inf')
        min_i, min_j = -1, -1
        for i in range(m):
            for j in range(n):
                if S[i] > 0 and D[j] > 0 and C[i][j] < min_cost:
                    min_cost = C[i][j]
                    min_i, min_j = i, j
        if min_i == -1:
            break
        allocation = min(S[min_i], D[min_j])
        X[min_i, min_j] = allocation
        total_cost += allocation * C[min_i][min_j]
        S[min_i] -= allocation
        D[min_j] -= allocation
        used += 1
        if S[min_i] == 0:
            for j in range(n):
                C[min_i][j] = float('inf')
        if D[min_j] == 0:
            for i in range(m):
                C[i][min_j] = float('inf')
    need = m + n - 1
    return X, total_cost, {"occupied": used, "required": need, "degenerate": used < need}
if __name__ == "__main__":
    try:
        supply = list(map(int, input("Supply (comma-separated): ").split(",")))
        demand = list(map(int, input("Demand (comma-separated): ").split(",")))
        m_initial = len(supply)
        n_initial = len(demand)
        print(f"Enter {m_initial} rows of cost matrix:")
        costs = [list(map(int, input().split(","))) for _ in range(m_initial)]
        #22CS8011 #Sankeerth
        X, z, info = least_cost_method(supply, demand, costs)
        print("\nAllocation Matrix:\n", X)
        print(f"\nTotal Cost: {z}")
        print("Degenerate" if info["degenerate"] else "Non-degenerate")
    except Exception as e:
        print("Error:", e)

In [ ]:
#4
import numpy as np
# 22CS8011 S Sankeerth Reddy
num_supply_points = int(input("Enter the number of supply points: "))
supply = np.array([int(x) for x in input(f"Enter supply quantities for {num_supply_points} points separated by spaces: ").split()])
num_demand_points = int(input("Enter the number of demand points: "))
demand = np.array([int(x) for x in input(f"Enter demand quantities for {num_demand_points} points separated by spaces: ").split()])
print(f"Enter the cost matrix ({num_supply_points}x{num_demand_points}):")
cost_matrix = []
for i in range(num_supply_points):
    row = [int(x) for x in input(f"Enter costs for row {i+1} separated by spaces: ").split()]
    cost_matrix.append(row)
cost_matrix = np.array(cost_matrix)
import numpy as np

def vam(cost, supply, demand):
    r, c = cost.shape
    alloc = np.zeros((r, c), int)
    s, d, cost_copy = supply.copy(), demand.copy(), cost.astype(float).copy()

    while s.sum() > 0 and d.sum() > 0:
        row_pen = [ (np.partition(cost_copy[i][d>0],1)[1]-np.partition(cost_copy[i][d>0],1)[0])
                    if s[i]>0 and (d>0).sum()>1 else (cost_copy[i][d>0][0] if s[i]>0 and (d>0).sum()==1 else 0)
                    for i in range(r) ]
        col_pen = [ (np.partition(cost_copy[:,j][s>0],1)[1]-np.partition(cost_copy[:,j][s>0],1)[0])
                    if d[j]>0 and (s>0).sum()>1 else (cost_copy[:,j][s>0][0] if d[j]>0 and (s>0).sum()==1 else 0)
                    for j in range(c) ]

        if max(row_pen) >= max(col_pen):
            i = np.argmax(row_pen); j = np.argmin(cost_copy[i])
        else:
            j = np.argmax(col_pen); i = np.argmin(cost_copy[:,j])

        qty = min(s[i], d[j]); alloc[i,j] = qty
        s[i] -= qty; d[j] -= qty
        if s[i]==0: cost_copy[i,:]=np.inf
        if d[j]==0: cost_copy[:,j]=np.inf
    return alloc

# 22CS8011 S Sankeerth Reddy
initial_alloc = vam(cost_matrix, supply, demand)
print("Initial BFS (Allocation Matrix):"); display(initial_alloc)
print(f"\nTotal Transportation Cost: {np.sum(initial_alloc*cost_matrix)}")


In [ ]:
#5
import numpy as np

def northwest_corner(supply, demand):
    s, d = supply.copy(), demand.copy()
    m, n = len(s), len(d)
    a = np.zeros((m, n))
    i = j = 0
    while i < m and j < n:
        x = min(s[i], d[j])
        a[i, j] = x
        s[i] -= x
        d[j] -= x
        if s[i] == 0 and d[j] == 0:
            if i < m - 1: i += 1
            elif j < n - 1: j += 1
            else: break
        elif s[i] == 0: i += 1
        else: j += 1
    return a

def ensure_basis(a, r):
    m, n = a.shape
    b = a > 0
    c = b.sum()
    if c < r:
        for i in range(m):
            for j in range(n):
                if not b[i, j]:
                    b[i, j] = True
                    c += 1
                if c == r: return b
    return b

def compute_uv(cost, basic):
    m, n = cost.shape
    u, v = [None] * m, [None] * n
    u[0] = 0.0
    ch = True
    while ch:
        ch = False
        for i in range(m):
            for j in range(n):
                if not basic[i, j]: continue
                if u[i] is not None and v[j] is None:
                    v[j] = cost[i, j] - u[i]; ch = True
                elif v[j] is not None and u[i] is None:
                    u[i] = cost[i, j] - v[j]; ch = True
    return u, v

def find_loop(basic_pos, start, m, n):
    allowed = set(basic_pos); allowed.add(start)
    def dfs(path, vis, move_row):
        i, j = path[-1]
        if move_row:
            for jj in range(n):
                if jj != j and (i, jj) in allowed:
                    if (i, jj) == start and len(path) >= 4: return path + [start]
                    if (i, jj) not in vis:
                        vis.add((i, jj))
                        r = dfs(path + [(i, jj)], vis, not move_row)
                        if r: return r
                        vis.remove((i, jj))
        else:
            for ii in range(m):
                if ii != i and (ii, j) in allowed:
                    if (ii, j) == start and len(path) >= 4: return path + [start]
                    if (ii, j) not in vis:
                        vis.add((ii, j))
                        r = dfs(path + [(ii, j)], vis, not move_row)
                        if r: return r
                        vis.remove((ii, j))
    for mv in (True, False):
        r = dfs([start], {start}, mv)
        if r: return r[:-1]
    return None

def modi_method(cost, supply, demand):
    cost, s, d = np.array(cost, float), np.array(supply, float), np.array(demand, float)
    if abs(s.sum() - d.sum()) > 1e-8: raise ValueError("Unbalanced problem")
    a = northwest_corner(s.tolist(), d.tolist())
    b = ensure_basis(a, cost.shape[0] + cost.shape[1] - 1)
    while True:
        u, v = compute_uv(cost, b)
        delta = np.full_like(cost, None, dtype=object)
        for i in range(cost.shape[0]):
            for j in range(cost.shape[1]):
                if not b[i, j] and u[i] is not None and v[j] is not None:
                    delta[i, j] = cost[i, j] - (u[i] + v[j])
        neg = [(i, j, delta[i, j]) for i in range(cost.shape[0]) for j in range(cost.shape[1]) if delta[i, j] is not None and delta[i, j] < 0]
        if not neg: break
        i0, j0, _ = min(neg, key=lambda x: x[2])
        loop = find_loop([(i, j) for i in range(cost.shape[0]) for j in range(cost.shape[1]) if b[i, j]], (i0, j0), *cost.shape)
        minus = [loop[k] for k in range(1, len(loop), 2)]
        theta = min(a[i, j] for i, j in minus if a[i, j] > 1e-10)
        for k, (i, j) in enumerate(loop):
            a[i, j] += theta if k % 2 == 0 else -theta
        b[i0, j0] = True
        for i, j in minus:
            if abs(a[i, j]) < 1e-8 and (i, j) != (i0, j0): b[i, j] = False
    return a.astype(int), int((a * cost).sum())

if __name__ == "__main__":
    m, n = int(input("Sources: ")), int(input("Destinations: "))
    print("Cost matrix:"); cost = [list(map(float, input().split())) for _ in range(m)]
    supply = list(map(float, input("Supply: ").split()))
    demand = list(map(float, input("Demand: ").split()))
    alloc, cost = modi_method(cost, supply, demand)
    print("\nFinal Allocation:\n", alloc)
    print("Minimum Cost:", cost)


In [ ]:
#6
def hungarian(a):
    n, m = len(a), len(a[0]); tr = False
    if n > m:
        a = [list(r) for r in zip(*a)]; n, m, tr = m, n, True
    u, v, p, way = [0]*(n+1), [0]*(m+1), [0]*(m+1), [0]*(m+1)
    INF = (lambda: 10**18)()
    for i in range(1, n+1):
        p[0] = i; minv, used = [INF]*(m+1), [False]*(m+1); j0 = 0
        while True:
            used[j0] = True; i0, delta, j1 = p[j0], INF, 0
            for j in range(1, m+1):
                if not used[j]:
                    cur = a[i0-1][j-1] - u[i0] - v[j]
                    if cur < minv[j]: minv[j], way[j] = cur, j0
                    if minv[j] < delta: delta, j1 = minv[j], j
            for j in range(m+1):
                if used[j]: u[p[j]] += delta; v[j] -= delta
                else: minv[j] -= delta
            j0 = j1
            if p[j0] == 0: break
        while True:
            j1 = way[j0]; p[j0] = p[j1]; j0 = j1
            if j0 == 0: break
    match = [-1]*n
    for j in range(1, m+1):
        if 1 <= p[j] <= n: match[p[j]-1] = j-1
    if tr:
        inv = [-1]*m
        for i, j in enumerate(match):
            if j != -1: inv[j] = i
        match = [inv[i] for i in range(m) if inv[i] != -1]
    return match
rin, rlist = (lambda: int(input().strip())), (lambda: list(map(int, input().split())))
n = rin()
m = rin()
a = [rlist() for _ in range(n)]
assign = hungarian(a)
total = sum(a[i][assign[i]] for i in range(len(assign)))
print(*((i+1, assign[i]+1) for i in range(len(assign))))
print(total)

In [ ]:
#7
def print_matrix(matrix):
    for row in matrix:
        print(row)
    print()

def step1_row_reduction(cost):
    n = len(cost)
    for i in range(n):
        min_val = cost[i][0]
        for j in range(len(cost[i])):
            if cost[i][j] < min_val:
                min_val = cost[i][j]
        for j in range(len(cost[i])):
            cost[i][j] -= min_val
    return cost

def step2_col_reduction(cost):
    n = len(cost[0])
    for j in range(n):
        min_val = cost[0][j]
        for i in range(len(cost)):
            if cost[i][j] < min_val:
                min_val = cost[i][j]
        for i in range(len(cost)):
            cost[i][j] -= min_val
    return cost

def cover_zeros(matrix):
    n = len(matrix)
    m = len(matrix[0])
    row_covered = [False] * n
    col_covered = [False] * m
    marked = [[0]*m for _ in range(n)]

    for i in range(n):
        for j in range(m):
            if matrix[i][j] == 0 and not row_covered[i] and not col_covered[j]:
                marked[i][j] = 1 # Star
                row_covered[i] = True
                col_covered[j] = True

    row_covered = [False] * n
    col_covered = [False] * m

    while True:
        for i in range(n):
            for j in range(m):
                if marked[i][j] == 1:
                    col_covered[j] = True

        count = sum(col_covered)
        if count == min(n, m):
            return marked

        while True:
            zero_found = False
            z_row, z_col = -1, -1
            for i in range(n):
                if not row_covered[i]:
                    for j in range(m):
                        if matrix[i][j] == 0 and not col_covered[j]:
                            z_row, z_col = i, j
                            zero_found = True
                            break
                    if zero_found:
                        break

            if not zero_found:
                min_val = float('inf')
                for i in range(n):
                    if not row_covered[i]:
                        for j in range(m):
                            if not col_covered[j]:
                                if matrix[i][j] < min_val:
                                    min_val = matrix[i][j]
                for i in range(n):
                    for j in range(m):
                        if row_covered[i] and col_covered[j]:
                            matrix[i][j] += min_val
                        elif not row_covered[i] and not col_covered[j]:
                            matrix[i][j] -= min_val
                continue
            else:
                marked[z_row][z_col] = 2 # Prime
                star_col = -1
                for j in range(m):
                    if marked[z_row][j] == 1:
                        star_col = j
                        break
                if star_col != -1:
                    row_covered[z_row] = True
                    col_covered[star_col] = False
                else:
                    path = [(z_row, z_col)]
                    while True:
                        row = -1
                        for i in range(n):
                            if marked[i][path[-1][1]] == 1:
                                row = i
                                break
                        if row == -1:
                            break
                        path.append((row, path[-1][1]))
                        col = -1
                        for j in range(m):
                            if marked[path[-1][0]][j] == 2:
                                col = j
                                break
                        path.append((path[-1][0], col))
                    for r, c in path:
                        if marked[r][c] == 1:
                            marked[r][c] = 0
                        elif marked[r][c] == 2:
                            marked[r][c] = 1
                    for i in range(n):
                        for j in range(m):
                            if marked[i][j] == 2:
                                marked[i][j] = 0
                    row_covered = [False] * n
                    col_covered = [False] * m
                    break

def get_assignments(marked):
    n = len(marked)
    m = len(marked[0])
    result = []
    for i in range(n):
        for j in range(m):
            if marked[i][j] == 1:
                result.append((i, j))
    return result

def pad_matrix(matrix, size):
    """Pad matrix with large values (e.g., 99999) to make it size x size"""
    n = len(matrix)
    m = len(matrix[0])
    padded = [row[:] + [99999]*(size - m) for row in matrix]
    for _ in range(size - n):
        padded.append([99999]*size)
    return padded

def hungarian_method(cost_matrix):
    n = len(cost_matrix)
    m = len(cost_matrix[0])
    size = max(n, m)

    # Pad to square matrix
    cost = pad_matrix(cost_matrix, size)

    cost = step1_row_reduction(cost)
    cost = step2_col_reduction(cost)
    marked = cover_zeros(cost)
    assignments = get_assignments(marked)

    # Filter out dummy assignments (cost=99999)
    real_assignments = [(i,j) for i,j in assignments if i < n and j < m]

    total_cost = sum(cost_matrix[i][j] for i, j in real_assignments)

    return real_assignments, total_cost

# --- USER INPUT ---

print("Enter number of workers (rows):")
m = int(input())

print("Enter number of jobs (columns):")
n = int(input())

print(f"Enter the cost matrix of size {m}x{n} (each row separated by Enter, values separated by spaces):")

cost_matrix = []
for i in range(m):
    while True:
        try:
            row = input(f"Row {i+1}: ").strip().split()
            if len(row) != n:
                print(f"Please enter exactly {n} numbers.")
                continue
            row = [int(x) for x in row]
            cost_matrix.append(row)
            break
        except ValueError:
            print("Invalid input, please enter integers only.")

assignments, total_cost = hungarian_method(cost_matrix)

print("\nOptimal Assignments (Worker -> Job):")
for worker, job in assignments:
    print(f"Worker {worker + 1} -> Job {job + 1} (Cost: {cost_matrix[worker][job]})")

print(f"\nTotal Minimum Cost: {total_cost}")